# AV2 - Etapa 1: ingestão e tokenização
**Grupo 2 - Responsável: Diego Bispo**

Este notebook é independente das células da AV1. No Colab, use **Ambiente de
execução > Executar tudo**. Os artefatos da AV1 são apenas lidos.

- Tokenizer: tiktoken 0.14.0, `cl100k_base`, `encode_ordinary`.
- Entrada: corpus e candidatos completos da AV1 (sem corte Top-k).
- Saída: instâncias de mochila 0/1 com W = 512, 1024, 2048, 4096.
- Não executa ainda os seletores guloso/PD e não chama uma LLM.

W é o orçamento dos textos dos chunks. A integração reservará espaço adicional
para instruções, consulta, citações e resposta do modelo.


In [1]:
import json
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/diegofnf/PAA_UFS_2026_2_Bispo_Diego_Marques_Gabriel_Andrade_Laryssa_Santos_Kaio_Melo_Victor.git'
try:
    import google.colab
    NO_COLAB = True
except ImportError:
    NO_COLAB = False
REPO_DIR = Path('/content/PAA_AV2') if NO_COLAB else Path.cwd()
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', 'av2', REPO_URL, str(REPO_DIR)], check=True)
if not (REPO_DIR / 'av2/1_scripts/preparar_instancias.py').is_file():
    raise RuntimeError('Use uma copia da branch av2; a main da AV1 nao contem esta etapa.')
print('Python em execucao:', sys.executable)
print('Commit em execucao:', subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(REPO_DIR / 'av2/requirements.txt'), 'pytest==9.1.1', 'nltk==3.10.3'], check=True)


Python em execucao: C:\Users\victo.000\AppData\Local\Programs\Python\Python314\python.exe
Commit em execucao: 6b815ca62cd72c13d7f166f44241d34787d0d6a6


CompletedProcess(args=['C:\\Users\\victo.000\\AppData\\Local\\Programs\\Python\\Python314\\python.exe', '-m', 'pip', 'install', '-q', '-r', 'C:\\Users\\victo.000\\AppData\\Local\\Temp\\paa_av2_clean\\av2\\requirements.txt', 'pytest==9.1.1', 'nltk==3.10.3'], returncode=0)

In [2]:
subprocess.run([sys.executable, str(REPO_DIR / 'av2/1_scripts/preparar_instancias.py')],
               cwd=REPO_DIR, check=True)
relatorio = json.loads((REPO_DIR / 'av2/5_resultados/relatorio_preparacao.json').read_text(encoding='utf-8'))
print('Consulta:', relatorio['metadados']['consulta'])
print('Tokenizer:', relatorio['metadados']['tokenizer'])
print('Chunks:', relatorio['total_chunks'], '| Candidatos positivos:', relatorio['total_candidatos_positivos'])
for instancia in relatorio['instancias']:
    print(instancia)
print('Tempos de preparacao (s):', relatorio['tempos_segundos'])


Consulta: critérios para atribuição de bolsas e requisitos de matrícula
Tokenizer: {'biblioteca': 'tiktoken', 'versao': '0.14.0', 'encoding': 'cl100k_base', 'metodo': 'encode_ordinary', 'modelo_llm': None}
Chunks: 182 | Candidatos positivos: 75
{'id_instancia': 'W_512', 'W': 512, 'total_itens': 75, 'total_itens_acima_W': 1}
{'id_instancia': 'W_1024', 'W': 1024, 'total_itens': 75, 'total_itens_acima_W': 0}
{'id_instancia': 'W_2048', 'W': 2048, 'total_itens': 75, 'total_itens_acima_W': 0}
{'id_instancia': 'W_4096', 'W': 4096, 'total_itens': 75, 'total_itens_acima_W': 0}
Tempos de preparacao (s): {'leitura_entradas_io': 0.027852300001541153, 'validacao_associacao_ram': 0.0005830000009154901, 'carregamento_tokenizer': 0.19219160001375712, 'tokenizacao_ram': 0.04267999999865424, 'montagem_instancias_ram': 0.00010910000128205866, 'serializacao_gravacao_dados_io': 0.007456999999703839, 'total_antes_gravar_relatorio': 0.6374648999917554}


In [3]:
subprocess.run([sys.executable, '-m', 'pytest', 'av2/1_scripts/test_preparar_instancias.py',
                'av1/1_scripts/test_etapa_4.py', '-q',
                '--junitxml=av2/6_evidencias/testes.xml'], cwd=REPO_DIR, check=True)
instancias = json.loads((REPO_DIR / 'av2/3_instancias/instancias.json').read_text(encoding='utf-8'))
assert [i['W'] for i in instancias['instancias']] == [512, 1024, 2048, 4096]
assert all(c['w_i'] > 0 and c['v_i'] > 0 for c in instancias['itens'])
print('Instancias prontas para as etapas de selecao. Veja av2/README.md para o contrato.')


Instancias prontas para as etapas de selecao. Veja av2/README.md para o contrato.


## AV2 - Etapa 2: Seleção de Contexto por Baseline Guloso (Mochila 0/1)
**Grupo 2 - Responsável: Kaio Santos**

Esta etapa consome as instâncias preparadas em `av2/3_instancias/instancias.json` e aplica
a heurística gulosa de densidade de relevância por token ($r_i = v_i / w_i$).

- **Objetivo:** Maximizar a relevância total $\sum v_i x_i$ sujeita a $\sum w_i x_i \le W$ com $x_i \in \{0, 1\}$.
- **Desempate determinístico:** maior densidade $r_i$ (desc), maior $v_i$ (desc), menor $w_i$ (asc), menor `id_chunk` (asc).
- **Varredura contínua:** Se um item não couber no saldo, o algoritmo o pula mas continua avaliando os itens subsequentes menores.
- **Saídas:** `av2/4_selecao_contexto/selecao_gulosa.json` e `av2/5_resultados/relatorio_guloso.json`.


In [4]:
subprocess.run([sys.executable, str(REPO_DIR / 'av2/1_scripts/selecionar_guloso.py')],
               cwd=REPO_DIR, check=True)
relatorio_guloso = json.loads((REPO_DIR / 'av2/5_resultados/relatorio_guloso.json').read_text(encoding='utf-8'))
print('Consulta:', relatorio_guloso['metadados']['consulta'])
print('Total de candidatos avaliados:', relatorio_guloso['metadados']['total_candidatos_positivos'])
print('\nResultados da Seleção Gulosa:')
for cenario in relatorio_guloso['resumo_cenarios']:
    print(f"  [W={cenario['W']:4d}] Chunks: {cenario['total_chunks_selecionados']:2d} | "
          f"Tokens: {cenario['tokens_usados']:4d}/{cenario['W']:4d} (Saldo: {cenario['saldo_W']:3d}, Ocupação: {cenario['taxa_ocupacao_percentual']:5.1f}%) | "
          f"Relevância: {cenario['relevancia_total']:7.4f} | IDs: {cenario['ids_selecionados']}")
print('\nTempos de execução da seleção (s):', relatorio_guloso['tempos_segundos'])


Consulta: critérios para atribuição de bolsas e requisitos de matrícula
Total de candidatos avaliados: 75

Resultados da Seleção Gulosa:
  [W= 512] Chunks:  1 | Tokens:  370/ 512 (Saldo: 142, Ocupação:  72.3%) | Relevância:  9.6206 | IDs: ['chunk_0131']
  [W=1024] Chunks:  2 | Tokens:  761/1024 (Saldo: 263, Ocupação:  74.3%) | Relevância: 16.6484 | IDs: ['chunk_0131', 'chunk_0018']
  [W=2048] Chunks:  5 | Tokens: 1905/2048 (Saldo: 143, Ocupação:  93.0%) | Relevância: 33.8168 | IDs: ['chunk_0131', 'chunk_0018', 'chunk_0176', 'chunk_0022', 'chunk_0139']
  [W=4096] Chunks: 10 | Tokens: 3820/4096 (Saldo: 276, Ocupação:  93.3%) | Relevância: 57.5855 | IDs: ['chunk_0131', 'chunk_0018', 'chunk_0176', 'chunk_0022', 'chunk_0139', 'chunk_0119', 'chunk_0178', 'chunk_0136', 'chunk_0066', 'chunk_0138']

Tempos de execução da seleção (s): {'selecao_gulosa_ram': 0.00040079999598674476, 'tempo_processamento_total': 0.00045760000648442656, 'leitura_io': 0.0020859999931417406, 'escrita_io': 0.0021126999

In [5]:
subprocess.run([sys.executable, str(REPO_DIR / 'av2/1_scripts/test_selecionar_guloso.py')],
               cwd=REPO_DIR, check=True)
selecao_gulosa = json.loads((REPO_DIR / 'av2/4_selecao_contexto/selecao_gulosa.json').read_text(encoding='utf-8'))
assert all(c['tokens_usados'] <= c['W'] for c in selecao_gulosa['cenarios'].values())
print('Todos os testes do seletor guloso passaram com sucesso!')


Todos os testes do seletor guloso passaram com sucesso!


## AV2 - Etapa 3: Seleção Ótima de Contexto por Programação Dinâmica (Mochila 0/1)
**Grupo 2 - Responsáveis: Diego Bispo e Kaio Santos**

Esta etapa resolve o problema da Mochila 0/1 exata via Programação Dinâmica bottom-up,
garantindo a seleção ótima global que maximiza a relevância total sem exceder $W$.

- **Equação de Bellman:** $DP[i][c] = \max(DP[i-1][c],\; DP[i-1][c - w_i] + v_i)$
- **Recuperação de Chunks:** Backtracking determinístico a partir de $DP[n][W]$.
- **Complexidade no Modelo RAM:** Tempo $\mathcal{O}(n \cdot W)$ e Espaço $\mathcal{O}(n \cdot W)$.
- **Saídas:** `av2/4_selecao_contexto/selecao_pd.json` e `av2/5_resultados/relatorio_pd.json`.


In [6]:
subprocess.run([sys.executable, str(REPO_DIR / 'av2/1_scripts/selecionar_pd.py')],
               cwd=REPO_DIR, check=True)
relatorio_pd = json.loads((REPO_DIR / 'av2/5_resultados/relatorio_pd.json').read_text(encoding='utf-8'))
relatorio_g = json.loads((REPO_DIR / 'av2/5_resultados/relatorio_guloso.json').read_text(encoding='utf-8'))
map_g = {c['W']: c for c in relatorio_g['resumo_cenarios']}

print('=== Confronto: Baseline Guloso vs. Programação Dinâmica ===')
for c_pd in relatorio_pd['resumo_cenarios']:
    W = c_pd['W']
    cg = map_g[W]
    ganho_rel = c_pd['relevancia_total'] - cg['relevancia_total']
    ganho_pct = (ganho_rel / cg['relevancia_total'] * 100) if cg['relevancia_total'] > 0 else 0
    print(f"\n[Capacidade W = {W}]")
    print(f"  Guloso: Chunks={cg['total_chunks_selecionados']:2d} | Tokens={cg['tokens_usados']:4d}/{W} (Saldo: {cg['saldo_W']:3d}) | Relevância={cg['relevancia_total']:7.4f}")
    print(f"  PD:     Chunks={c_pd['total_chunks_selecionados']:2d} | Tokens={c_pd['tokens_usados']:4d}/{W} (Saldo: {c_pd['saldo_W']:3d}) | Relevância={c_pd['relevancia_total']:7.4f}")
    print(f"  Ganho da PD: {ganho_rel:+.4f} ({ganho_pct:+.2f}%)")
print('\nTempos de execução da PD (s):', relatorio_pd['tempos_segundos'])


=== Confronto: Baseline Guloso vs. Programação Dinâmica ===

[Capacidade W = 512]
  Guloso: Chunks= 1 | Tokens= 370/512 (Saldo: 142) | Relevância= 9.6206
  PD:     Chunks= 1 | Tokens= 370/512 (Saldo: 142) | Relevância= 9.6206
  Ganho da PD: +0.0000 (+0.00%)

[Capacidade W = 1024]
  Guloso: Chunks= 2 | Tokens= 761/1024 (Saldo: 263) | Relevância=16.6484
  PD:     Chunks= 2 | Tokens= 761/1024 (Saldo: 263) | Relevância=16.6484
  Ganho da PD: +0.0000 (+0.00%)

[Capacidade W = 2048]
  Guloso: Chunks= 5 | Tokens=1905/2048 (Saldo: 143) | Relevância=33.8168
  PD:     Chunks= 5 | Tokens=1905/2048 (Saldo: 143) | Relevância=33.8168
  Ganho da PD: +0.0000 (+0.00%)

[Capacidade W = 4096]
  Guloso: Chunks=10 | Tokens=3820/4096 (Saldo: 276) | Relevância=57.5855
  PD:     Chunks=11 | Tokens=4093/4096 (Saldo:   3) | Relevância=60.3138
  Ganho da PD: +2.7283 (+4.74%)

Tempos de execução da PD (s): {'selecao_pd_ram': 0.14097670000046492, 'tempo_processamento_total': 0.14103490000707097, 'leitura_io': 0.00

In [7]:
subprocess.run([sys.executable, str(REPO_DIR / 'av2/1_scripts/test_selecionar_pd.py')],
               cwd=REPO_DIR, check=True)
selecao_pd = json.loads((REPO_DIR / 'av2/4_selecao_contexto/selecao_pd.json').read_text(encoding='utf-8'))
assert all(c['tokens_usados'] <= c['W'] for c in selecao_pd['cenarios'].values())
print('Todos os testes da Programação Dinâmica passaram com sucesso!')


Todos os testes da Programação Dinâmica passaram com sucesso!


## Etapas seguintes
- **Etapa 4 — Benchmark Comparativo:** comparação entre guloso e PD nos quatro orçamentos.
- **Etapa 5 — Contexto para LLM:** montagem do prompt, orçamento de tokens, deduplicação e análise de latência/qualidade.
- **Avaliação semântica (trabalho futuro):** envio dos payloads a um LLM com perguntas de referência.


## AV2 - Etapa 4: Benchmark comparativo e evidências

Esta etapa compara os resultados da heurística gulosa com a solução ótima por
programação dinâmica nas quatro capacidades. A comparação considera tokens
utilizados, ocupação, relevância acumulada, ganho relativo e tempos registrados
pelos seletores.

A etapa também executa a suíte de testes da AV2 e preserva a evidência JUnit.


In [8]:
subprocess.run([sys.executable, str(REPO_DIR / 'av2/1_scripts/gerar_relatorio_corretude.py')],
               cwd=REPO_DIR, check=True)
relatorio_corretude = json.loads((REPO_DIR / 'av2/5_resultados/relatorio_corretude_limites.json').read_text(encoding='utf-8'))
relatorio_guloso = json.loads((REPO_DIR / 'av2/5_resultados/relatorio_guloso.json').read_text(encoding='utf-8'))
relatorio_pd = json.loads((REPO_DIR / 'av2/5_resultados/relatorio_pd.json').read_text(encoding='utf-8'))
guloso_por_w = {cenario['W']: cenario for cenario in relatorio_guloso['resumo_cenarios']}

print('=== Etapa 4: Corretude e benchmark Guloso vs. PD ===')
for cenario_pd in relatorio_pd['resumo_cenarios']:
    capacidade = cenario_pd['W']
    cenario_guloso = guloso_por_w[capacidade]
    ganho = cenario_pd['relevancia_total'] - cenario_guloso['relevancia_total']
    assert cenario_pd['tokens_usados'] <= capacidade
    assert cenario_pd['relevancia_total'] >= cenario_guloso['relevancia_total'] - 1e-4
    print(f"W={capacidade}: "
          f"guloso={cenario_guloso['relevancia_total']:.4f}, "
          f"PD={cenario_pd['relevancia_total']:.4f}, "
          f"ganho={ganho:+.4f}, "
          f"tokens_PD={cenario_pd['tokens_usados']}/{capacidade}")

subprocess.run([sys.executable, '-m', 'pytest',
                'av2/1_scripts/test_corretude_limites.py', '-q'],
               cwd=REPO_DIR, check=True)
print('Relatório, contraexemplo e testes específicos da Etapa 4 executados com sucesso.')
print('Os artefatos das etapas anteriores foram reutilizados sem reexecução.')


=== Etapa 4: Corretude e benchmark Guloso vs. PD ===
W=512: guloso=9.6206, PD=9.6206, ganho=+0.0000, tokens_PD=370/512
W=1024: guloso=16.6484, PD=16.6484, ganho=+0.0000, tokens_PD=761/1024
W=2048: guloso=33.8168, PD=33.8168, ganho=+0.0000, tokens_PD=1905/2048
W=4096: guloso=57.5855, PD=60.3138, ganho=+2.7283, tokens_PD=4093/4096


Relatório, contraexemplo e testes específicos da Etapa 4 executados com sucesso.
Os artefatos das etapas anteriores foram reutilizados sem reexecução.


## AV2 - Etapa 5: contexto organizado para LLM

Esta etapa lê as seleções gulosa e PD, verifica os custos `w_i` contra
`tiktoken` (`cl100k_base`, `encode_ordinary`), monta o prompt com fontes
`[F1]...`, deduplica a sobreposição de 30 palavras e recontar o prompt final.
`W` é tratado como orçamento de seleção, não como janela da LLM: instruções,
resposta e segurança são reservadas separadamente.

A análise completa está em `av2/CONTEXTO_LLM.md`; os payloads ficam em
`av2/7_contexto_llm/contextos.json`.

In [9]:
subprocess.run([sys.executable, str(REPO_DIR / 'av2/integrar_contexto.py'),
                '--repeticoes', '5'], cwd=REPO_DIR, check=True)
relatorio_contexto = json.loads((REPO_DIR / 'av2/5_resultados/relatorio_contexto_llm.json').read_text(encoding='utf-8'))

assert relatorio_contexto['validacao_w_i']['total_chunks'] == 182
assert relatorio_contexto['validacao_w_i']['tokens_individuais_total'] == 74055
for item in relatorio_contexto['resumo_por_orcamento']:
    assert item['tokens_prompt'] >= item['tokens_w_i']

print('=== Etapa 5: contexto para LLM ===')
for item in relatorio_contexto['resumo_por_orcamento']:
    print(f"{item['seletor']} W={item['W']}: "
          f"prompt={item['tokens_prompt']} tokens, "
          f"overhead={item['overhead_prompt']}, "
          f"relevância={item['relevancia_incluida']:.4f}, "
          f"folga={item['folga_llm']}")

subprocess.run([sys.executable, '-m', 'pytest',
                'av2/1_scripts/test_montar_contexto_llm.py', '-q'],
               cwd=REPO_DIR, check=True)
print('Contexto, tabela, gráficos e testes da Etapa 5 executados com sucesso.')

=== Etapa 5: contexto para LLM ===
guloso W=512: prompt=607 tokens, overhead=237, relevância=9.6206, folga=7009
guloso W=1024: prompt=1033 tokens, overhead=271, relevância=16.6484, folga=6583
guloso W=2048: prompt=2304 tokens, overhead=396, relevância=33.8168, folga=5312
guloso W=4096: prompt=4394 tokens, overhead=623, relevância=57.5855, folga=3222
pd W=512: prompt=607 tokens, overhead=237, relevância=9.6206, folga=7009
pd W=1024: prompt=1033 tokens, overhead=271, relevância=16.6484, folga=6583
pd W=2048: prompt=2304 tokens, overhead=396, relevância=33.8168, folga=5312
pd W=4096: prompt=4712 tokens, overhead=667, relevância=60.3138, folga=2904


Contexto, tabela, gráficos e testes da Etapa 5 executados com sucesso.
